In [21]:
!pip install pandas numpy scikit-learn xgboost statsmodels pmdarima joblib -q

In [22]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print(name)

Saving blood_bags_discarded.csv to blood_bags_discarded.csv
Saving blood_bags_issued.csv to blood_bags_issued.csv
Saving blood_bank_directory.csv to blood_bank_directory.csv
Saving blood_bank_population_ratio.csv to blood_bank_population_ratio.csv
Saving blood_units_collected.csv to blood_units_collected.csv
Saving donation_camps.csv to donation_camps.csv
Saving licensed_blood_banks.csv to licensed_blood_banks.csv
Saving smartblood_synthetic_data.csv to smartblood_synthetic_data.csv

Uploaded files:
blood_bags_discarded.csv
blood_bags_issued.csv
blood_bank_directory.csv
blood_bank_population_ratio.csv
blood_units_collected.csv
donation_camps.csv
licensed_blood_banks.csv
smartblood_synthetic_data.csv


In [23]:
FILES = {
    "directory": "blood_bank_directory.csv",
    "camps": "donation_camps.csv",
    "licensed": "licensed_blood_banks.csv",
    "collected": "blood_units_collected.csv",
    "issued": "blood_bags_issued.csv",
    "discarded": "blood_bags_discarded.csv",
    "ratio": "blood_bank_population_ratio.csv",
    "smartblood": "smartblood_synthetic_data.csv"
}

In [24]:
import pandas as pd
import numpy as np
import os

data = {}

for key, path in FILES.items():
    data[key] = pd.read_csv(path)
    print(f"{key}: {data[key].shape}")

directory_df = data["directory"]
camps_df = data["camps"]
licensed_df = data["licensed"]
collected_df = data["collected"]
issued_df = data["issued"]
discarded_df = data["discarded"]
ratio_df = data["ratio"]
smartblood_df = data["smartblood"]

directory: (2823, 27)
camps: (37, 4)
licensed: (37, 6)
collected: (38, 6)
issued: (38, 5)
discarded: (37, 3)
ratio: (37, 4)
smartblood: (210528, 14)


In [52]:
import re
from pathlib import Path

OUTPUT_DIR = Path("")
OUTPUT_DIR.mkdir(exist_ok=True)

RNG = np.random.default_rng(42)

def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df

def num(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace(",", "").strip()
    x = re.sub(r"[^\d.\-]", "", x)
    return pd.to_numeric(x, errors="coerce")

def normalize_state(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()

    aliases = {
        "andaman and nicobar islands": "Andaman and Nicobar Islands",
        "andaman nicobar islands": "Andaman and Nicobar Islands",
        "andhra pradesh": "Andhra Pradesh",
        "arunachal pradesh": "Arunachal Pradesh",
        "assam": "Assam",
        "bihar": "Bihar",
        "chandigarh": "Chandigarh",
        "chhattisgarh": "Chhattisgarh",
        "chattisgarh": "Chhattisgarh",
        "dadra and nagar haveli": "Dadra and Nagar Haveli and Daman and Diu",
        "daman and diu": "Dadra and Nagar Haveli and Daman and Diu",
        "delhi": "Delhi",
        "nct of delhi": "Delhi",
        "goa": "Goa",
        "gujarat": "Gujarat",
        "haryana": "Haryana",
        "himachal pradesh": "Himachal Pradesh",
        "jammu and kashmir": "Jammu and Kashmir",
        "jammu kashmir": "Jammu and Kashmir",
        "jharkhand": "Jharkhand",
        "karnataka": "Karnataka",
        "kerala": "Kerala",
        "ladakh": "Ladakh",
        "lakshadweep": "Lakshadweep",
        "madhya pradesh": "Madhya Pradesh",
        "maharashtra": "Maharashtra",
        "manipur": "Manipur",
        "meghalaya": "Meghalaya",
        "mizoram": "Mizoram",
        "nagaland": "Nagaland",
        "odisha": "Odisha",
        "orissa": "Odisha",
        "puducherry": "Puducherry",
        "pondicherry": "Puducherry",
        "punjab": "Punjab",
        "rajasthan": "Rajasthan",
        "sikkim": "Sikkim",
        "tamil nadu": "Tamil Nadu",
        "telangana": "Telangana",
        "tripura": "Tripura",
        "uttar pradesh": "Uttar Pradesh",
        "uttarakhand": "Uttarakhand",
        "uttaranchal": "Uttarakhand",
        "west bengal": "West Bengal"
    }

    return aliases.get(s, str(x).strip())

for key in data:
    data[key] = clean_columns(data[key])

directory_df = data["directory"]
camps_df = data["camps"]
licensed_df = data["licensed"]
collected_df = data["collected"]
issued_df = data["issued"]
discarded_df = data["discarded"]
ratio_df = data["ratio"]
smartblood_df = data["smartblood"]

In [26]:
for name, df in data.items():
    print(f"\n{'=' * 70}")
    print(f"{name.upper()}")
    print(f"{'=' * 70}")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nFirst 3 rows:")
    print(df.head(3))


DIRECTORY
Shape: (2823, 27)

Columns:
['Sr No', 'Blood Bank Name', 'State', 'District', 'City', 'Address', 'Pincode', 'Contact No', 'Mobile', 'Helpline', 'Fax', 'Email', 'Website', 'Nodal Officer', 'Contact Nodal Officer', 'Mobile Nodal Officer', 'Email Nodal Officer', 'Qualification Nodal Officer', 'Category', 'Blood Component Available', 'Apheresis', 'Service Time', 'License #', 'Date License Obtained', 'Date of Renewal', 'Latitude', 'Longitude']

Data types:
Sr No                            int64
Blood Bank Name                 object
State                           object
District                        object
City                            object
Address                         object
Pincode                         object
Contact No                      object
Mobile                          object
Helpline                        object
Fax                             object
Email                           object
Website                         object
Nodal Officer             

In [27]:
smart = smartblood_df.copy()

smart["date"] = pd.to_datetime(smart["date"], errors="coerce")

print("Date range:", smart["date"].min(), "to", smart["date"].max())
print("Total rows:", len(smart))
print("Unique blood banks:", smart["blood_bank_id"].nunique())
print("Unique blood groups:", smart["blood_group"].nunique())
print("Blood groups:", sorted(smart["blood_group"].unique()))
print("Component types:", sorted(smart["component_type"].unique()))

print("\nRecords per blood bank:")
print(smart["blood_bank_id"].value_counts().describe())

print("\nRecords per component:")
print(smart["component_type"].value_counts())

print("\nRecords per blood group:")
print(smart["blood_group"].value_counts())

print("\nMissing dates:", smart["date"].isna().sum())

Date range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00
Total rows: 210528
Unique blood banks: 12
Unique blood groups: 8
Blood groups: ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
Component types: ['FFP', 'PRBC', 'Platelets']

Records per blood bank:
count       12.0
mean     17544.0
std          0.0
min      17544.0
25%      17544.0
50%      17544.0
75%      17544.0
max      17544.0
Name: count, dtype: float64

Records per component:
component_type
PRBC         70176
Platelets    70176
FFP          70176
Name: count, dtype: int64

Records per blood group:
blood_group
O+     26316
B+     26316
A+     26316
AB+    26316
O-     26316
B-     26316
A-     26316
AB-    26316
Name: count, dtype: int64

Missing dates: 0


In [28]:
smart["expected_closing"] = (
    smart["opening_inventory"]
    + smart["donations_collected"]
    + smart["transfers_in"]
    - smart["transfers_out"]
    - smart["units_fulfilled"]
    - smart["units_discarded_tti"]
    - smart["units_expired"]
    - smart["units_wasted"]
)

smart["inventory_difference"] = (
    smart["closing_inventory"] - smart["expected_closing"]
)

print("Inventory reconciliation:")
print(smart["inventory_difference"].value_counts().sort_index())

print("\nRows with inventory mismatch:",
      (smart["inventory_difference"] != 0).sum())

print("\nNegative values:")
numeric_cols = [
    "opening_inventory",
    "donations_collected",
    "units_discarded_tti",
    "transfers_in",
    "transfers_out",
    "units_requested",
    "units_fulfilled",
    "units_expired",
    "units_wasted",
    "closing_inventory"
]

print((smart[numeric_cols] < 0).sum())

Inventory reconciliation:
inventory_difference
0      196044
1        5438
2        2775
3        1362
4         998
        ...  
75          1
76          1
77          1
78          1
112         1
Name: count, Length: 70, dtype: int64

Rows with inventory mismatch: 14484

Negative values:
opening_inventory      0
donations_collected    0
units_discarded_tti    0
transfers_in           0
transfers_out          0
units_requested        0
units_fulfilled        0
units_expired          0
units_wasted           0
closing_inventory      0
dtype: int64


In [29]:
smart = smart.sort_values(
    ["blood_bank_id", "blood_group", "component_type", "date"]
).reset_index(drop=True)

group_cols = ["blood_bank_id", "blood_group", "component_type"]

print("Duplicate records:",
      smart.duplicated(group_cols + ["date"]).sum())

print("\nDate gaps:")
dates = smart.groupby(group_cols)["date"].diff().dt.days
print(dates.value_counts().sort_index())

print("\nRows per series:")
series_counts = smart.groupby(group_cols).size()
print(series_counts.describe())

print("\nNumber of unique series:", len(series_counts))
print("Expected:", 12 * 8 * 3)

Duplicate records: 0

Date gaps:
date
1.0    210240
Name: count, dtype: int64

Rows per series:
count    288.0
mean     731.0
std        0.0
min      731.0
25%      731.0
50%      731.0
75%      731.0
max      731.0
dtype: float64

Number of unique series: 288
Expected: 288


In [30]:
print("Demand statistics:")
print(smart["units_requested"].describe())

print("\nFulfillment statistics:")
print(smart["units_fulfilled"].describe())

print("\nInventory statistics:")
print(smart["closing_inventory"].describe())

print("\nRequest > Fulfilled:")
print(
    (smart["units_requested"] > smart["units_fulfilled"]).value_counts()
)

print("\nFulfillment rate:")
smart["fulfillment_rate"] = np.where(
    smart["units_requested"] > 0,
    smart["units_fulfilled"] / smart["units_requested"],
    1.0
)

print(smart["fulfillment_rate"].describe())

print("\nRows where demand exceeded fulfillment:",
      (smart["units_requested"] > smart["units_fulfilled"]).sum())

Demand statistics:
count    210528.000000
mean          3.344054
std           6.207484
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max         145.000000
Name: units_requested, dtype: float64

Fulfillment statistics:
count    210528.000000
mean          3.275726
std           6.128235
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max         145.000000
Name: units_fulfilled, dtype: float64

Inventory statistics:
count    210528.000000
mean         79.361971
std         145.599087
min           0.000000
25%           5.000000
50%          23.000000
75%          75.000000
max        1315.000000
Name: closing_inventory, dtype: float64

Request > Fulfilled:
False    205643
True       4885
Name: count, dtype: int64

Fulfillment rate:
count    210528.000000
mean          0.982810
std           0.120787
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max  

In [31]:
smart = smart.sort_values(
    ["blood_bank_id", "blood_group", "component_type", "date"]
).reset_index(drop=True)

group_cols = ["blood_bank_id", "blood_group", "component_type"]

smart["day_of_week"] = smart["date"].dt.dayofweek
smart["month"] = smart["date"].dt.month
smart["day_of_year"] = smart["date"].dt.dayofyear
smart["week_of_year"] = smart["date"].dt.isocalendar().week.astype(int)

smart["is_weekend"] = (smart["day_of_week"] >= 5).astype(int)

g = smart.groupby(group_cols, group_keys=False)

smart["demand_lag_1"] = g["units_requested"].shift(1)
smart["demand_lag_7"] = g["units_requested"].shift(7)
smart["demand_lag_14"] = g["units_requested"].shift(14)
smart["demand_lag_28"] = g["units_requested"].shift(28)

smart["demand_avg_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["demand_avg_14"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(14).mean()
)

smart["demand_avg_28"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(28).mean()
)

smart["demand_std_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).std()
)

smart["fulfillment_avg_7"] = g["fulfillment_rate"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["inventory_avg_7"] = g["closing_inventory"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["inventory_lag_1"] = g["closing_inventory"].shift(1)
smart["inventory_lag_7"] = g["closing_inventory"].shift(7)

smart["donations_avg_7"] = g["donations_collected"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["transfers_in_avg_7"] = g["transfers_in"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["transfers_out_avg_7"] = g["transfers_out"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["wastage_avg_7"] = g["units_wasted"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

smart["expiry_avg_7"] = g["units_expired"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

print("Feature engineering complete.")
print("Shape:", smart.shape)

print("\nNew columns:")
new_cols = [
    c for c in smart.columns
    if c not in smartblood_df.columns
]
print(new_cols)

Feature engineering complete.
Shape: (210528, 39)

New columns:
['expected_closing', 'inventory_difference', 'fulfillment_rate', 'day_of_week', 'month', 'day_of_year', 'week_of_year', 'is_weekend', 'demand_lag_1', 'demand_lag_7', 'demand_lag_14', 'demand_lag_28', 'demand_avg_7', 'demand_avg_14', 'demand_avg_28', 'demand_std_7', 'fulfillment_avg_7', 'inventory_avg_7', 'inventory_lag_1', 'inventory_lag_7', 'donations_avg_7', 'transfers_in_avg_7', 'transfers_out_avg_7', 'wastage_avg_7', 'expiry_avg_7']


In [32]:
smart["target_demand_next_day"] = g["units_requested"].shift(-1)

print("Target created.")

print("\nTarget statistics:")
print(smart["target_demand_next_day"].describe())

print("\nMissing target values:",
      smart["target_demand_next_day"].isna().sum())

Target created.

Target statistics:
count    210240.000000
mean          3.343764
std           6.205614
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max         145.000000
Name: target_demand_next_day, dtype: float64

Missing target values: 288


In [33]:
feature_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type",
    "day_of_week",
    "month",
    "day_of_year",
    "week_of_year",
    "is_weekend",
    "demand_lag_1",
    "demand_lag_7",
    "demand_lag_14",
    "demand_lag_28",
    "demand_avg_7",
    "demand_avg_14",
    "demand_avg_28",
    "demand_std_7",
    "fulfillment_avg_7",
    "inventory_avg_7",
    "inventory_lag_1",
    "inventory_lag_7",
    "donations_avg_7",
    "transfers_in_avg_7",
    "transfers_out_avg_7",
    "wastage_avg_7",
    "expiry_avg_7"
]

target_col = "target_demand_next_day"

model_df = smart[
    feature_cols + ["date", target_col]
].dropna().copy()

print("ML dataset shape:", model_df.shape)
print("\nMissing values:")
print(model_df.isna().sum().sum())

print("\nDate range:")
print(model_df["date"].min(), "to", model_df["date"].max())

ML dataset shape: (202176, 27)

Missing values:
0

Date range:
2023-01-29 00:00:00 to 2024-12-30 00:00:00


In [34]:
TRAIN_END = pd.Timestamp("2024-09-30")
TEST_START = pd.Timestamp("2024-10-01")

train_df = model_df[model_df["date"] <= TRAIN_END].copy()
test_df = model_df[model_df["date"] >= TEST_START].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTesting period:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nTrain target mean:", train_df[target_col].mean())
print("Test target mean:", test_df[target_col].mean())

Training rows: 175968
Testing rows: 26208

Training period:
2023-01-29 00:00:00 to 2024-09-30 00:00:00

Testing period:
2024-10-01 00:00:00 to 2024-12-30 00:00:00

Train target mean: 3.3749545371885796
Test target mean: 3.1842567155067156


In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_predictions = test_df["demand_avg_7"]

baseline_mae = mean_absolute_error(
    test_df[target_col],
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        test_df[target_col],
        baseline_predictions
    )
)

print("7-Day Moving Average Baseline")
print("--------------------------------")
print("MAE :", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))

7-Day Moving Average Baseline
--------------------------------
MAE : 2.0991
RMSE: 4.3467


In [36]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type"
]

numeric_cols = [
    c for c in feature_cols
    if c not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "numeric",
            "passthrough",
            numeric_cols
        )
    ]
)

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

Encoded training shape: (175968, 45)
Encoded testing shape: (26208, 45)


# **MODEL TRAINING**

In [37]:
from sklearn.ensemble import HistGradientBoostingRegressor

X_train_dense = X_train_encoded
X_test_dense = X_test_encoded

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

print("Training model...")

model.fit(X_train_dense, y_train)

print("Training complete.")

Training model...
Training complete.


In [38]:
ml_predictions = model.predict(X_test_dense)

ml_predictions = np.maximum(
    ml_predictions,
    0
)

ml_mae = mean_absolute_error(
    y_test,
    ml_predictions
)

ml_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        ml_predictions
    )
)

print("ML Model Performance")
print("--------------------")
print("MAE :", round(ml_mae, 4))
print("RMSE:", round(ml_rmse, 4))

print("\nBaseline Performance")
print("--------------------")
print("MAE :", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))

ML Model Performance
--------------------
MAE : 2.0259
RMSE: 4.127

Baseline Performance
--------------------
MAE : 2.0991
RMSE: 4.3467


Analyze Prediction Errors

In [39]:
results = test_df[
    ["date", "blood_bank_id", "blood_group", "component_type", target_col]
].copy()

results["predicted_demand"] = ml_predictions
results["baseline_prediction"] = baseline_predictions.values

results["ml_error"] = (
    results[target_col] - results["predicted_demand"]
)

results["absolute_error"] = (
    results[target_col] - results["predicted_demand"]
).abs()

results["baseline_absolute_error"] = (
    results[target_col] - results["baseline_prediction"]
).abs()

results["improved_over_baseline"] = (
    results["absolute_error"] <
    results["baseline_absolute_error"]
)

print(results.head(10))

          date blood_bank_id blood_group component_type  \
639 2024-10-01         BB001          A+            FFP   
640 2024-10-02         BB001          A+            FFP   
641 2024-10-03         BB001          A+            FFP   
642 2024-10-04         BB001          A+            FFP   
643 2024-10-05         BB001          A+            FFP   
644 2024-10-06         BB001          A+            FFP   
645 2024-10-07         BB001          A+            FFP   
646 2024-10-08         BB001          A+            FFP   
647 2024-10-09         BB001          A+            FFP   
648 2024-10-10         BB001          A+            FFP   

     target_demand_next_day  predicted_demand  baseline_prediction  ml_error  \
639                     2.0          4.471884             5.000000 -2.471884   
640                     2.0          4.514877             4.428571 -2.514877   
641                     7.0          4.567222             4.428571  2.432778   
642                     2.0   

In [40]:
print("Overall ML MAE:",
      results["absolute_error"].mean())

print("Overall baseline MAE:",
      results["baseline_absolute_error"].mean())

print("\nML better than baseline:")
print(
    results["improved_over_baseline"].mean() * 100,
    "%"
)

Overall ML MAE: 2.025930949957182
Overall baseline MAE: 2.0991027821384964

ML better than baseline:
46.38278388278388 %


In [41]:
group_error = (
    results
    .groupby("blood_group")
    .agg(
        observations=(target_col, "size"),
        actual_mean=(target_col, "mean"),
        predicted_mean=("predicted_demand", "mean"),
        mae=("absolute_error", "mean")
    )
    .sort_values("mae", ascending=False)
)

print(group_error)

             observations  actual_mean  predicted_mean       mae
blood_group                                                     
O+                   3276     9.827228       10.222513  5.513062
B+                   3276     6.078755        6.050929  3.731441
A+                   3276     5.644383        5.677252  3.515053
AB+                  3276     1.795482        1.838841  1.475633
O-                   3276     0.960623        0.975869  0.673557
B-                   3276     0.559524        0.594996  0.529364
A-                   3276     0.484127        0.530698  0.489846
AB-                  3276     0.123932        0.230306  0.279493


In [42]:
component_error = (
    results
    .groupby("component_type")
    .agg(
        observations=(target_col, "size"),
        actual_mean=(target_col, "mean"),
        predicted_mean=("predicted_demand", "mean"),
        mae=("absolute_error", "mean")
    )
    .sort_values("mae", ascending=False)
)

print(component_error)

                observations  actual_mean  predicted_mean       mae
component_type                                                     
PRBC                    8736     5.262134        5.432119  3.278209
Platelets               8736     2.422734        2.405912  1.534445
FFP                     8736     1.867903        1.957495  1.265139


In [43]:
bank_error = (
    results
    .groupby("blood_bank_id")
    .agg(
        observations=(target_col, "size"),
        actual_mean=(target_col, "mean"),
        predicted_mean=("predicted_demand", "mean"),
        mae=("absolute_error", "mean")
    )
    .sort_values("mae", ascending=False)
)

print(bank_error)

               observations  actual_mean  predicted_mean       mae
blood_bank_id                                                     
BB001                  2184     3.840659        3.960582  2.474047
BB003                  2184     3.963370        3.950298  2.469126
BB005                  2184     3.908883        3.911123  2.460906
BB007                  2184     3.902930        3.991396  2.454267
BB006                  2184     3.881868        3.990770  2.435270
BB010                  2184     3.865385        3.952476  2.397050
BB004                  2184     3.767399        3.902765  2.394810
BB009                  2184     3.984890        3.985297  2.393448
BB002                  2184     2.151099        2.250136  1.446943
BB008                  2184     2.100733        2.197168  1.388051
BB011                  2184     2.005037        2.125184  1.366241
BB012                  2184     0.838828        0.964911  0.631012


In [44]:
high_demand = results[
    results[target_col] >= 10
].copy()

print("High-demand observations:", len(high_demand))

if len(high_demand) > 0:
    print(
        "High-demand MAE:",
        round(high_demand["absolute_error"].mean(), 4)
    )

    print(
        "High-demand RMSE:",
        round(
            np.sqrt(
                np.mean(high_demand["ml_error"] ** 2)
            ),
            4
        )
    )

    print("\nWorst predictions:")
    print(
        high_demand
        .sort_values("absolute_error", ascending=False)
        .head(15)
        [
            [
                "date",
                "blood_bank_id",
                "blood_group",
                "component_type",
                target_col,
                "predicted_demand",
                "absolute_error"
            ]
        ]
    )

High-demand observations: 2401
High-demand MAE: 8.1402
High-demand RMSE: 10.9489

Worst predictions:
             date blood_bank_id blood_group component_type  \
71611  2024-12-05         BB005          A+           PRBC   
154940 2024-11-30         BB009          O+           PRBC   
14543  2024-10-16         BB001          O+           PRBC   
119872 2024-12-20         BB007          O+           PRBC   
45247  2024-10-18         BB003          B+           PRBC   
154945 2024-12-05         BB009          O+           PRBC   
62790  2024-10-17         BB004          B+           PRBC   
154939 2024-11-29         BB009          O+           PRBC   
154954 2024-12-14         BB009          O+           PRBC   
154957 2024-12-17         BB009          O+           PRBC   
45261  2024-11-01         BB003          B+           PRBC   
102269 2024-10-22         BB006          O+           PRBC   
84723  2024-10-20         BB005          O+           PRBC   
119796 2024-10-05         BB007

In [45]:
spike_cases = results.nlargest(10, "absolute_error")

for _, row in spike_cases.iterrows():
    bank = row["blood_bank_id"]
    group = row["blood_group"]
    component = row["component_type"]
    date = row["date"]

    history = smart[
        (smart["blood_bank_id"] == bank) &
        (smart["blood_group"] == group) &
        (smart["component_type"] == component) &
        (smart["date"] >= date - pd.Timedelta(days=7)) &
        (smart["date"] <= date + pd.Timedelta(days=1))
    ][
        [
            "date",
            "units_requested",
            "units_fulfilled",
            "opening_inventory",
            "donations_collected",
            "transfers_in",
            "transfers_out",
            "closing_inventory"
        ]
    ]

    print("\n" + "=" * 80)
    print(
        f"{bank} | {group} | {component} | "
        f"Spike date: {date.date()}"
    )
    print("=" * 80)
    print(history.to_string(index=False))


BB005 | A+ | PRBC | Spike date: 2024-12-05
      date  units_requested  units_fulfilled  opening_inventory  donations_collected  transfers_in  transfers_out  closing_inventory
2024-11-28               14               14                328                    7             0              0                321
2024-11-29                6                6                321                    2             0              2                315
2024-11-30                8                8                315                    2             0              0                309
2024-12-01               12               12                309                    5             0              0                302
2024-12-02               31               31                302                   20             0              0                291
2024-12-03                6                6                291                   16             0              5                296
2024-12-04               

In [46]:
spikes = smart[
    smart["units_requested"] >= 10
].copy()

print("Total high-demand rows:", len(spikes))

print("\nBy blood group:")
print(
    spikes.groupby("blood_group")
    .size()
    .sort_values(ascending=False)
)

print("\nBy component:")
print(
    spikes.groupby("component_type")
    .size()
    .sort_values(ascending=False)
)

print("\nBy blood bank:")
print(
    spikes.groupby("blood_bank_id")
    .size()
    .sort_values(ascending=False)
)

Total high-demand rows: 20713

By blood group:
blood_group
O+     9976
B+     5465
A+     4669
AB+     596
O-        7
dtype: int64

By component:
component_type
PRBC         13093
Platelets     4686
FFP           2934
dtype: int64

By blood bank:
blood_bank_id
BB003    2294
BB005    2268
BB007    2267
BB006    2252
BB010    2250
BB001    2234
BB004    2205
BB009    2192
BB002     890
BB008     886
BB011     871
BB012     104
dtype: int64


In [47]:
spikes["month"] = spikes["date"].dt.month

print("\nHigh-demand events by month:")
print(
    spikes.groupby("month")
    .size()
    .sort_index()
)


High-demand events by month:
month
1     1698
2     1582
3     1711
4     1687
5     1930
6     1820
7     1946
8     1828
9     1647
10    1637
11    1303
12    1924
dtype: int64


In [48]:
seasonality = (
    smart.groupby("day_of_week")["units_requested"]
    .agg(["mean", "median", "max", "count"])
)

print("Demand by day of week:")
print(seasonality)

print("\nDemand by month:")
print(
    smart.groupby("month")["units_requested"]
    .agg(["mean", "median", "max", "count"])
)

Demand by day of week:
                 mean  median  max  count
day_of_week                              
0            3.338029     1.0   90  30240
1            3.348049     1.0  108  30240
2            3.416533     1.0   86  29952
3            3.383747     1.0   96  29952
4            3.402778     1.0   95  29952
5            3.320446     1.0  145  29952
6            3.200198     1.0  100  30240

Demand by month:
           mean  median  max  count
month                              
1      3.264113     1.0   81  17856
2      3.299647     1.0  100  16416
3      3.284890     1.0  145  17856
4      3.332407     1.0   95  17280
5      3.595318     1.0   79  17856
6      3.491319     1.0  108  17280
7      3.640121     1.0   98  17856
8      3.421763     1.0   87  17856
9      3.255382     1.0   88  17280
10     3.208333     1.0   80  17856
11     2.740625     1.0   76  17280
12     3.573197     1.0   93  17856


In [49]:
smart["target_date"] = smart["date"] + pd.Timedelta(days=1)

smart["target_day_of_week"] = smart["target_date"].dt.dayofweek
smart["target_month"] = smart["target_date"].dt.month
smart["target_day_of_year"] = smart["target_date"].dt.dayofyear
smart["target_week_of_year"] = smart["target_date"].dt.isocalendar().week.astype(int)
smart["target_is_weekend"] = (smart["target_day_of_week"] >= 5).astype(int)

In [50]:
feature_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type",
    "target_day_of_week",
    "target_month",
    "target_day_of_year",
    "target_week_of_year",
    "target_is_weekend",
    "demand_lag_1",
    "demand_lag_7",
    "demand_lag_14",
    "demand_lag_28",
    "demand_avg_3",
    "demand_avg_7",
    "demand_avg_14",
    "demand_avg_28",
    "demand_max_7",
    "demand_max_14",
    "demand_max_28",
    "demand_std_7",
    "spike_count_7",
    "spike_count_14",
    "spike_count_28",
    "demand_trend",
    "fulfillment_avg_7",
    "inventory_avg_7",
    "inventory_lag_1",
    "inventory_lag_7",
    "donations_avg_7",
    "transfers_in_avg_7",
    "transfers_out_avg_7",
    "wastage_avg_7",
    "expiry_avg_7"
]

target_col = "target_demand_next_day"

In [51]:
model_df = smart[feature_cols + ["date", target_col]].dropna().copy()

train_df = model_df[model_df["date"] <= TRAIN_END].copy()
test_df = model_df[model_df["date"] >= TEST_START].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

KeyError: "['demand_avg_3', 'demand_max_7', 'demand_max_14', 'demand_max_28', 'spike_count_7', 'spike_count_14', 'spike_count_28', 'demand_trend'] not in index"

In [ ]:
g = smart.groupby(group_cols, group_keys=False)

smart["demand_avg_3"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)

smart["demand_max_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).max()
)

smart["demand_max_14"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(14).max()
)

smart["demand_max_28"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(28).max()
)

smart["spike_count_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).apply(lambda y: (y >= 10).sum())
)

smart["spike_count_14"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(14).apply(lambda y: (y >= 10).sum())
)

smart["spike_count_28"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(28).apply(lambda y: (y >= 10).sum())
)

smart["demand_trend"] = (
    smart["demand_avg_3"] - smart["demand_avg_14"]
)

In [ ]:
smart["target_date"] = smart["date"] + pd.Timedelta(days=1)

smart["target_day_of_week"] = smart["target_date"].dt.dayofweek
smart["target_month"] = smart["target_date"].dt.month
smart["target_day_of_year"] = smart["target_date"].dt.dayofyear
smart["target_week_of_year"] = smart["target_date"].dt.isocalendar().week.astype(int)
smart["target_is_weekend"] = (smart["target_day_of_week"] >= 5).astype(int)

In [ ]:
g = smart.groupby(group_cols, group_keys=False)

smart["demand_avg_3"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)

smart["demand_max_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).max()
)

smart["demand_max_14"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(14).max()
)

smart["demand_max_28"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(28).max()
)

smart["spike_count_7"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(7).apply(lambda y: (y >= 10).sum())
)

smart["spike_count_14"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(14).apply(lambda y: (y >= 10).sum())
)

smart["spike_count_28"] = g["units_requested"].transform(
    lambda x: x.shift(1).rolling(28).apply(lambda y: (y >= 10).sum())
)

smart["demand_trend"] = smart["demand_avg_3"] - smart["demand_avg_14"]

In [ ]:
model_df = smart[feature_cols + ["date", target_col]].dropna().copy()

train_df = model_df[model_df["date"] <= TRAIN_END].copy()
test_df = model_df[model_df["date"] >= TEST_START].copy()

print("Model dataset:", model_df.shape)
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Missing values:", model_df.isna().sum().sum())

In [ ]:
categorical_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type"
]

numeric_cols = [
    c for c in feature_cols
    if c not in categorical_cols
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", "passthrough", numeric_cols)
    ]
)

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train_encoded, y_train)

ml_predictions = model.predict(X_test_encoded)
ml_predictions = np.maximum(ml_predictions, 0)

ml_mae = mean_absolute_error(y_test, ml_predictions)
ml_rmse = np.sqrt(mean_squared_error(y_test, ml_predictions))

print("New ML MAE:", ml_mae)
print("New ML RMSE:", ml_rmse)

print("\nPrevious ML MAE: 2.0259")
print("Previous ML RMSE: 4.1270")

print("\nBaseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

In [ ]:
print("MODEL COMPARISON")
print("=" * 40)

print(f"Baseline MAE:  {baseline_mae:.4f}")
print(f"New ML MAE:    {ml_mae:.4f}")

print()

print(f"Baseline RMSE: {baseline_rmse:.4f}")
print(f"New ML RMSE:   {ml_rmse:.4f}")

mae_improvement = ((baseline_mae - ml_mae) / baseline_mae) * 100
rmse_improvement = ((baseline_rmse - ml_rmse) / baseline_rmse) * 100

print()
print(f"MAE improvement:  {mae_improvement:.2f}%")
print(f"RMSE improvement: {rmse_improvement:.2f}%")

In [ ]:
results = test_df[
    ["date", "blood_bank_id", "blood_group", "component_type", target_col]
].copy()

results["predicted_demand"] = ml_predictions
results["baseline_prediction"] = baseline_predictions.values

results["absolute_error"] = (
    results[target_col] - results["predicted_demand"]
).abs()

results["baseline_absolute_error"] = (
    results[target_col] - results["baseline_prediction"]
).abs()

results["improved_over_baseline"] = (
    results["absolute_error"] < results["baseline_absolute_error"]
)

print("Overall ML MAE:", results["absolute_error"].mean())
print("Baseline MAE:", results["baseline_absolute_error"].mean())
print(
    "ML better than baseline:",
    results["improved_over_baseline"].mean() * 100,
    "%"
)

In [ ]:
high_demand = results[results[target_col] >= 10].copy()

print("High-demand observations:", len(high_demand))
print("High-demand MAE:", high_demand["absolute_error"].mean())
print("High-demand RMSE:", np.sqrt(
    mean_squared_error(
        high_demand[target_col],
        high_demand["predicted_demand"]
    )
))

print("\nWorst predictions:")
print(
    high_demand.nlargest(15, "absolute_error")[
        [
            "date",
            "blood_bank_id",
            "blood_group",
            "component_type",
            target_col,
            "predicted_demand",
            "absolute_error"
        ]
    ].to_string(index=False)
)

In [ ]:
smart["target_spike"] = (
    smart["target_demand_next_day"] >= 10
).astype(int)

print(
    smart["target_spike"].value_counts()
)

print(
    smart["target_spike"].value_counts(normalize=True) * 100
)

In [ ]:
spike_feature_cols = feature_cols.copy()

spike_df = smart[
    spike_feature_cols + ["date", "target_spike"]
].dropna().copy()

spike_train = spike_df[
    spike_df["date"] <= TRAIN_END
].copy()

spike_test = spike_df[
    spike_df["date"] >= TEST_START
].copy()

print("Spike dataset:", spike_df.shape)
print("Training rows:", len(spike_train))
print("Testing rows:", len(spike_test))

In [ ]:
categorical_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type"
]

numeric_cols = [
    c for c in spike_feature_cols
    if c not in categorical_cols
]

spike_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        (
            "numeric",
            "passthrough",
            numeric_cols
        )
    ]
)

X_spike_train = spike_train[spike_feature_cols]
y_spike_train = spike_train["target_spike"]

X_spike_test = spike_test[spike_feature_cols]
y_spike_test = spike_test["target_spike"]

X_spike_train_encoded = spike_preprocessor.fit_transform(
    X_spike_train
)

X_spike_test_encoded = spike_preprocessor.transform(
    X_spike_test
)

print(
    "Training shape:",
    X_spike_train_encoded.shape
)

print(
    "Testing shape:",
    X_spike_test_encoded.shape
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

spike_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

spike_model.fit(
    X_spike_train_encoded,
    y_spike_train
)

spike_probabilities = spike_model.predict_proba(
    X_spike_test_encoded
)[:, 1]

spike_predictions = (
    spike_probabilities >= 0.5
).astype(int)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print(
    classification_report(
        y_spike_test,
        spike_predictions,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_spike_test,
        spike_predictions
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_spike_test,
        spike_probabilities
    )
)

In [ ]:
spike_results = spike_test[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "target_spike"
    ]
].copy()

spike_results["spike_probability"] = spike_probabilities

print("Probability for actual normal days:")
print(
    spike_results.loc[
        spike_results["target_spike"] == 0,
        "spike_probability"
    ].describe()
)

print("\nProbability for actual spike days:")
print(
    spike_results.loc[
        spike_results["target_spike"] == 1,
        "spike_probability"
    ].describe()
)

In [ ]:
print("\nHighest-probability normal observations:")
print(
    spike_results[
        spike_results["target_spike"] == 0
    ].nlargest(
        15,
        "spike_probability"
    ).to_string(index=False)
)

print("\nLowest-probability actual spikes:")
print(
    spike_results[
        spike_results["target_spike"] == 1
    ].nsmallest(
        15,
        "spike_probability"
    ).to_string(index=False)
)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70
]

threshold_results = []

for threshold in thresholds:
    predictions = (
        spike_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_spike_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_spike_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_spike_test,
            predictions,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

Saving Model 1.
Model 1 — Final status
Demand Forecast Model

Purpose: Predict next-day demand.

Model: HistGradientBoostingRegressor
MAE: 2.0235
RMSE: 4.1174
Improvement vs 7-day moving average:
MAE: 3.60%
RMSE: 5.28%
Spike Risk Model

Purpose: Estimate whether next-day demand will be ≥10 units.

Model: HistGradientBoostingClassifier
ROC-AUC: 0.9384
At threshold 0.30:
Precision: 50.24%
Recall: 69.35%
F1: 58.27%

For an inventory-warning system, 0.30 is more useful than 0.50 because missing a genuine high-demand event is costly. But we should treat 0.30 as an experimental operating threshold, not as a universally optimal threshold.

One important finding

The classifier's probabilities aren't perfectly calibrated.

For example, some actual normal days receive probabilities above 0.80, while some actual spikes receive probabilities below 0.01.

So we should not tell the user:

"There is an 82% chance of a spike."

as if that were a statistically calibrated probability.

For now, the safer interpretation is:

"The model assigns a high spike-risk score."

That distinction matters when you present this project.

In [ ]:
import joblib

joblib.dump(model, OUTPUT_DIR / "demand_forecast_model.pkl")
joblib.dump(preprocessor, OUTPUT_DIR / "demand_forecast_preprocessor.pkl")

joblib.dump(spike_model, OUTPUT_DIR / "spike_risk_model.pkl")
joblib.dump(spike_preprocessor, OUTPUT_DIR / "spike_risk_preprocessor.pkl")

print("Model 1 files saved:")
print("demand_forecast_model.pkl")
print("demand_forecast_preprocessor.pkl")
print("spike_risk_model.pkl")
print("spike_risk_preprocessor.pkl")

**MODEL 2 PREP**

In [ ]:
inventory_analysis = smart[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "units_requested",
        "units_fulfilled"
    ]
].copy()

inventory_analysis["next_day_demand"] = (
    inventory_analysis
    .groupby(group_cols)["units_requested"]
    .shift(-1)
)

inventory_analysis["inventory_coverage"] = (
    inventory_analysis["closing_inventory"] /
    inventory_analysis["next_day_demand"].replace(0, np.nan)
)

print("Inventory coverage statistics:")
print(
    inventory_analysis["inventory_coverage"].describe()
)

print("\nCoverage below 1:")
print(
    (inventory_analysis["inventory_coverage"] < 1).sum()
)

print(
    "Percentage:",
    (inventory_analysis["inventory_coverage"] < 1).mean() * 100
)

print("\nCoverage below 2:")
print(
    (inventory_analysis["inventory_coverage"] < 2).sum()
)

print(
    "Percentage:",
    (inventory_analysis["inventory_coverage"] < 2).mean() * 100
)

In [ ]:
print("\nLowest inventory coverage cases:")

print(
    inventory_analysis[
        inventory_analysis["next_day_demand"] > 0
    ]
    .nsmallest(20, "inventory_coverage")
    .to_string(index=False)
)

In [ ]:
shortage_cases = inventory_analysis[
    inventory_analysis["next_day_demand"] > 0
].copy()

shortage_cases["shortage_risk"] = (
    shortage_cases["closing_inventory"] <
    shortage_cases["next_day_demand"]
).astype(int)

print("Shortage target distribution:")
print(
    shortage_cases["shortage_risk"]
    .value_counts()
)

print("\nPercentages:")
print(
    shortage_cases["shortage_risk"]
    .value_counts(normalize=True) * 100
)

print("\nShortage cases:")
print(
    shortage_cases[
        shortage_cases["shortage_risk"] == 1
    ]
    .head(20)
    .to_string(index=False)
)

In [ ]:
shortage_summary = shortage_cases.groupby(
    "shortage_risk"
).agg(
    cases=("shortage_risk", "size"),
    avg_inventory=("closing_inventory", "mean"),
    avg_next_day_demand=("next_day_demand", "mean"),
    avg_fulfillment=("units_fulfilled", "mean"),
    avg_requested=("units_requested", "mean")
)

print(shortage_summary)

# **Building Model 2**

In [ ]:
inventory_model_df = smart[
    inventory_feature_cols +
    ["date", "closing_inventory"]
].copy()

inventory_model_df["next_day_demand"] = (
    smart
    .groupby(group_cols)["units_requested"]
    .shift(-1)
)

inventory_model_df["shortage_risk"] = (
    inventory_model_df["closing_inventory"] <
    inventory_model_df["next_day_demand"]
).astype(int)

inventory_model_df = inventory_model_df.dropna()

print("Inventory model dataset:", inventory_model_df.shape)

print("\nTarget distribution:")
print(
    inventory_model_df["shortage_risk"].value_counts()
)

print("\nTarget percentages:")
print(
    inventory_model_df["shortage_risk"]
    .value_counts(normalize=True) * 100
)

In [ ]:
inventory_train = inventory_model_df[
    inventory_model_df["date"] <= TRAIN_END
].copy()

inventory_test = inventory_model_df[
    inventory_model_df["date"] >= TEST_START
].copy()

X_inventory_train = inventory_train[inventory_feature_cols]
y_inventory_train = inventory_train["shortage_risk"]

X_inventory_test = inventory_test[inventory_feature_cols]
y_inventory_test = inventory_test["shortage_risk"]

print("Training set:", inventory_train.shape)
print("Test set:", inventory_test.shape)

print("\nTraining target:")
print(y_inventory_train.value_counts())

print("\nTest target:")
print(y_inventory_test.value_counts())

print("\nTest target percentages:")
print(
    y_inventory_test.value_counts(normalize=True) * 100
)

Model 2 features

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

inventory_categorical_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type"
]

inventory_numeric_cols = [
    col for col in inventory_feature_cols
    if col not in inventory_categorical_cols
]

inventory_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            inventory_categorical_cols
        ),
        (
            "numeric",
            "passthrough",
            inventory_numeric_cols
        )
    ]
)

X_inventory_train_encoded = inventory_preprocessor.fit_transform(
    X_inventory_train
)

X_inventory_test_encoded = inventory_preprocessor.transform(
    X_inventory_test
)

print("Encoded training shape:", X_inventory_train_encoded.shape)
print("Encoded test shape:", X_inventory_test_encoded.shape)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

class_counts = y_inventory_train.value_counts()

weight_0 = 1.0
weight_1 = class_counts[0] / class_counts[1]

inventory_sample_weights = y_inventory_train.map({
    0: weight_0,
    1: weight_1
})

inventory_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

inventory_model.fit(
    X_inventory_train_encoded,
    y_inventory_train,
    sample_weight=inventory_sample_weights
)

inventory_risk_scores = inventory_model.predict_proba(
    X_inventory_test_encoded
)[:, 1]

inventory_predictions = (
    inventory_risk_scores >= 0.5
).astype(int)

print("Model 2 training complete.")
print("Risk score range:", inventory_risk_scores.min(), "to", inventory_risk_scores.max())

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print("Classification Report:")
print(
    classification_report(
        y_inventory_test,
        inventory_predictions,
        digits=4
    )
)

cm = confusion_matrix(
    y_inventory_test,
    inventory_predictions
)

print("Confusion Matrix:")
print(cm)

roc_auc = roc_auc_score(
    y_inventory_test,
    inventory_risk_scores
)

pr_auc = average_precision_score(
    y_inventory_test,
    inventory_risk_scores
)

precision = precision_score(
    y_inventory_test,
    inventory_predictions
)

recall = recall_score(
    y_inventory_test,
    inventory_predictions
)

f1 = f1_score(
    y_inventory_test,
    inventory_predictions
)

print("\nROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

print("\nActual shortage cases:", y_inventory_test.sum())
print(
    "Predicted shortage cases:",
    inventory_predictions.sum()
)

In [ ]:
threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (
        inventory_risk_scores >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_inventory_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_inventory_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_inventory_test,
            predictions,
            zero_division=0
        ),
        "predicted_shortages": predictions.sum()
    })

threshold_df = pd.DataFrame(threshold_results)

print(
    threshold_df.to_string(index=False)
)

In [ ]:
inventory_results = inventory_test[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "next_day_demand",
        "shortage_risk"
    ]
].copy()

inventory_results["risk_score"] = inventory_risk_scores
inventory_results["predicted_shortage"] = inventory_predictions

inventory_results["correct"] = (
    inventory_results["shortage_risk"] ==
    inventory_results["predicted_shortage"]
)

print("Highest-risk predictions:")
print(
    inventory_results
    .sort_values("risk_score", ascending=False)
    .head(20)
    .to_string(index=False)
)

print("\nMissed shortage cases:")

missed = inventory_results[
    (inventory_results["shortage_risk"] == 1) &
    (inventory_results["predicted_shortage"] == 0)
]

print(
    missed
    .sort_values("risk_score", ascending=False)
    .head(20)
    .to_string(index=False)
)

In [ ]:
baseline_inventory_score = (
    inventory_test["closing_inventory"] <
    inventory_test["demand_avg_7"]
).astype(int)

print("Baseline inventory rule:")
print(
    classification_report(
        y_inventory_test,
        baseline_inventory_score,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_inventory_test,
        baseline_inventory_score
    )
)

print(
    "Baseline F1:",
    f1_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

print(
    "Baseline Recall:",
    recall_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

print(
    "Baseline Precision:",
    precision_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

In [ ]:
comparison = threshold_df.copy()

comparison["beats_baseline_f1"] = (
    comparison["f1"] >
    f1_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

comparison["beats_baseline_precision"] = (
    comparison["precision"] >
    precision_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

comparison["beats_baseline_recall"] = (
    comparison["recall"] >
    recall_score(
        y_inventory_test,
        baseline_inventory_score,
        zero_division=0
    )
)

print(comparison.to_string(index=False))

print("\nBest ML threshold by F1:")
print(
    comparison.loc[
        comparison["f1"].idxmax()
    ]
)

# **Model 2 conclusion**

The ML model does not outperform the simple baseline on this test set, even after threshold tuning.

The best ML threshold we tested was 0.85:

Precision: 40.04%
Recall: 65.10%
F1: 49.59%

The baseline is:

Precision: 44.14%
Recall: 82.38%
F1: 57.49%

So the baseline is better on precision, recall, and F1.

The ML model's ROC-AUC of 0.9606 is still interesting, but it isn't enough to justify replacing the simpler rule. In practical terms, the model can rank risk reasonably well, but its conversion into useful shortage alerts isn't beating the existing inventory-coverage logic.

Therefore freeze Model 2 as a supporting experiment, not as the primary shortage detector.

Our production architecture should use:

                 CURRENT INVENTORY
                        │
                        ▼
             ┌─────────────────────┐
             │ Inventory Coverage  │
             │                     │
             │ Inventory < Demand  │
             │       Average       │
             └──────────┬──────────┘
                        │
                        ▼
                 SHORTAGE SIGNAL
                        │
          ┌─────────────┴─────────────┐
          │                           │
          ▼                           ▼
    Demand Forecast-Model1    Spike Risk-Model1
          │                           │
          └─────────────┬─────────────┘
                        ▼
                DECISION ENGINE
                        │
            ┌───────────┼───────────┐
            ▼           ▼           ▼
          Normal      Warning     Critical


**Saving Model 2.**

In [ ]:
import joblib

joblib.dump(
    inventory_model,
    OUTPUT_DIR / "inventory_risk_model.pkl"
)

joblib.dump(
    inventory_preprocessor,
    OUTPUT_DIR / "inventory_risk_preprocessor.pkl"
)

print("Model 2 files saved:")
print("inventory_risk_model.pkl")
print("inventory_risk_preprocessor.pkl")

What we have now

Model 1 — Demand Intelligence

Next-day demand regression
Spike-risk classification
Handles ordinary demand + identifies unusually high-demand risk
Regression improvement over moving-average baseline:
MAE: 2.0235 vs 2.0991
RMSE: 4.1174 vs 4.3467

Model 2 — Inventory Intelligence

Tested ML shortage-risk classifier
Tested against simple inventory-coverage rule
Baseline performs better
Therefore primary inventory shortage detection remains rule-based

# **Model 3 — Decision & Risk Engine**

In [ ]:
decision_df = inventory_test[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "demand_avg_7"
    ]
].copy()

decision_df["predicted_demand"] = ml_predictions

spike_results = spike_test[
    ["date", "blood_bank_id", "blood_group", "component_type"]
].copy()

spike_results["spike_risk"] = spike_probabilities

decision_df = decision_df.merge(
    spike_results,
    on=["date", "blood_bank_id", "blood_group", "component_type"],
    how="left"
)

inventory_results = inventory_test[
    ["date", "blood_bank_id", "blood_group", "component_type"]
].copy()

inventory_results["inventory_risk"] = inventory_risk_scores

decision_df = decision_df.merge(
    inventory_results,
    on=["date", "blood_bank_id", "blood_group", "component_type"],
    how="left"
)

decision_df["predicted_demand"] = decision_df["predicted_demand"].clip(lower=0)

decision_df["predicted_coverage"] = (
    decision_df["closing_inventory"] /
    decision_df["predicted_demand"].replace(0, np.nan)
)

print(decision_df.shape)
decision_df.head()

In [ ]:
def get_risk(row):
    if row["predicted_demand"] > 0 and row["closing_inventory"] < row["predicted_demand"]:
        return "Critical"
    if row["predicted_coverage"] < 2 or row["spike_risk"] >= 0.30:
        return "High"
    if row["predicted_coverage"] < 5 or row["inventory_risk"] >= 0.50:
        return "Medium"
    return "Low"

decision_df["risk_level"] = decision_df.apply(get_risk, axis=1)

print(decision_df["risk_level"].value_counts())

In [ ]:
def get_action(row):
    if row["risk_level"] == "Critical":
        return "Urgent stock replenishment or transfer"
    if row["risk_level"] == "High":
        return "Monitor closely and prepare replenishment"
    if row["risk_level"] == "Medium":
        return "Monitor inventory"
    return "No immediate action"

decision_df["recommended_action"] = decision_df.apply(
    get_action,
    axis=1
)

print(
    decision_df[
        [
            "blood_bank_id",
            "blood_group",
            "component_type",
            "closing_inventory",
            "predicted_demand",
            "spike_risk",
            "predicted_coverage",
            "risk_level",
            "recommended_action"
        ]
    ].head(10)
)

In [ ]:
critical_cases = decision_df[
    decision_df["risk_level"] == "Critical"
].copy()

actual_results = inventory_test[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "shortage_risk"
    ]
].copy()

critical_cases = critical_cases.merge(
    actual_results,
    on=[
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    how="left"
)

print("Critical cases:", len(critical_cases))
print("\nActual shortage:")
print(critical_cases["shortage_risk"].value_counts())

print("\nCritical shortage rate:")
print(critical_cases["shortage_risk"].mean())

In [ ]:
risk_validation = decision_df.merge(
    inventory_test[
        [
            "date",
            "blood_bank_id",
            "blood_group",
            "component_type",
            "shortage_risk"
        ]
    ],
    on=[
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    how="left"
)

print(
    risk_validation.groupby("risk_level")["shortage_risk"]
    .agg(["count", "sum", "mean"])
)

In [ ]:
decision_df["risk_score"] = (
    0.50 * decision_df["inventory_risk"] +
    0.30 * decision_df["spike_risk"] +
    0.20 * (
        1 / decision_df["predicted_coverage"].clip(lower=1)
    )
)

decision_df["risk_score"] = decision_df["risk_score"].clip(0, 1)

print(
    decision_df[
        [
            "blood_bank_id",
            "blood_group",
            "component_type",
            "closing_inventory",
            "predicted_demand",
            "predicted_coverage",
            "spike_risk",
            "inventory_risk",
            "risk_score",
            "risk_level"
        ]
    ].head(10)
)

In [ ]:
risk_validation = decision_df.merge(
    inventory_test[
        [
            "date",
            "blood_bank_id",
            "blood_group",
            "component_type",
            "shortage_risk"
        ]
    ],
    on=[
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    how="left"
)

risk_validation["risk_band"] = pd.qcut(
    risk_validation["risk_score"],
    5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

print(
    risk_validation.groupby("risk_band", observed=True)["shortage_risk"]
    .agg(["count", "sum", "mean"])
)

In [ ]:
risk_thresholds = risk_validation["risk_score"].quantile(
    [0.20, 0.40, 0.60, 0.80]
).values

print("Risk thresholds:")
print(risk_thresholds)

In [ ]:
def assign_risk_band(score):
    if score <= risk_thresholds[0]:
        return "Very Low"
    if score <= risk_thresholds[1]:
        return "Low"
    if score <= risk_thresholds[2]:
        return "Medium"
    if score <= risk_thresholds[3]:
        return "High"
    return "Very High"

decision_df["risk_band"] = decision_df["risk_score"].apply(assign_risk_band)

print(decision_df["risk_band"].value_counts())

In [ ]:
print(
    decision_df[
        [
            "date",
            "blood_bank_id",
            "blood_group",
            "component_type",
            "closing_inventory",
            "predicted_demand",
            "predicted_coverage",
            "spike_risk",
            "inventory_risk",
            "risk_score",
            "risk_band",
            "recommended_action"
        ]
    ].head(10)
)

In [ ]:
wastage_analysis = smart[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "units_requested",
        "units_expired",
        "units_wasted"
    ]
].copy()

print(wastage_analysis[
    ["closing_inventory", "units_requested", "units_expired", "units_wasted"]
].describe())

In [ ]:
print("Total expired:", wastage_analysis["units_expired"].sum())
print("Total wasted:", wastage_analysis["units_wasted"].sum())

print("\nRows with expiry:")
print((wastage_analysis["units_expired"] > 0).sum())

print("\nRows with wastage:")
print((wastage_analysis["units_wasted"] > 0).sum())

In [ ]:
g = smart.groupby(group_cols, group_keys=False)

smart["next_day_wastage"] = g["units_wasted"].shift(-1)
smart["next_day_expiry"] = g["units_expired"].shift(-1)

smart["wastage_risk"] = (
    (smart["next_day_wastage"] > 0) |
    (smart["next_day_expiry"] > 0)
).astype(int)

print(smart["wastage_risk"].value_counts())
print(smart["wastage_risk"].value_counts(normalize=True))

In [ ]:
wastage_feature_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type",
    "target_day_of_week",
    "target_month",
    "target_day_of_year",
    "target_week_of_year",
    "target_is_weekend",
    "inventory_lag_1",
    "inventory_lag_7",
    "inventory_avg_7",
    "demand_lag_1",
    "demand_lag_7",
    "demand_lag_14",
    "demand_avg_3",
    "demand_avg_7",
    "demand_avg_14",
    "demand_avg_28",
    "demand_max_7",
    "demand_max_14",
    "demand_max_28",
    "demand_std_7",
    "spike_count_7",
    "spike_count_14",
    "spike_count_28",
    "demand_trend",
    "fulfillment_avg_7",
    "donations_avg_7",
    "transfers_in_avg_7",
    "transfers_out_avg_7"
]

wastage_model_df = smart[
    wastage_feature_cols +
    ["date", "wastage_risk"]
].dropna().copy()

print("Dataset shape:", wastage_model_df.shape)
print("\nTarget distribution:")
print(wastage_model_df["wastage_risk"].value_counts())

In [ ]:
wastage_train = wastage_model_df[
    wastage_model_df["date"] <= TRAIN_END
].copy()

wastage_test = wastage_model_df[
    wastage_model_df["date"] >= TEST_START
].copy()

X_wastage_train = wastage_train[wastage_feature_cols]
y_wastage_train = wastage_train["wastage_risk"]

X_wastage_test = wastage_test[wastage_feature_cols]
y_wastage_test = wastage_test["wastage_risk"]

print("Train:", X_wastage_train.shape)
print("Test:", X_wastage_test.shape)

print("\nTrain target:")
print(y_wastage_train.value_counts())

print("\nTest target:")
print(y_wastage_test.value_counts())

In [ ]:
wastage_categorical_cols = [
    "blood_bank_id",
    "blood_group",
    "component_type"
]

wastage_numeric_cols = [
    col for col in wastage_feature_cols
    if col not in wastage_categorical_cols
]

wastage_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            wastage_categorical_cols
        ),
        (
            "numeric",
            "passthrough",
            wastage_numeric_cols
        )
    ]
)

X_wastage_train_encoded = wastage_preprocessor.fit_transform(
    X_wastage_train
)

X_wastage_test_encoded = wastage_preprocessor.transform(
    X_wastage_test
)

print("Train encoded:", X_wastage_train_encoded.shape)
print("Test encoded:", X_wastage_test_encoded.shape)

In [ ]:
wastage_class_counts = y_wastage_train.value_counts()

wastage_weight_0 = 1.0
wastage_weight_1 = (
    wastage_class_counts[0] /
    wastage_class_counts[1]
)

wastage_sample_weights = y_wastage_train.map({
    0: wastage_weight_0,
    1: wastage_weight_1
})

wastage_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.08,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

wastage_model.fit(
    X_wastage_train_encoded,
    y_wastage_train,
    sample_weight=wastage_sample_weights
)

wastage_risk_scores = wastage_model.predict_proba(
    X_wastage_test_encoded
)[:, 1]

wastage_predictions = (
    wastage_risk_scores >= 0.5
).astype(int)

print(
    "Risk score range:",
    wastage_risk_scores.min(),
    "to",
    wastage_risk_scores.max()
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print(classification_report(
    y_wastage_test,
    wastage_predictions
))

print("Confusion matrix:")
print(confusion_matrix(
    y_wastage_test,
    wastage_predictions
))

print("ROC-AUC:", roc_auc_score(
    y_wastage_test,
    wastage_risk_scores
))

print("PR-AUC:", average_precision_score(
    y_wastage_test,
    wastage_risk_scores
))

print("Precision:", precision_score(
    y_wastage_test,
    wastage_predictions
))

print("Recall:", recall_score(
    y_wastage_test,
    wastage_predictions
))

print("F1:", f1_score(
    y_wastage_test,
    wastage_predictions
))

In [ ]:
baseline_wastage_predictions = (
    wastage_test["inventory_lag_1"] >
    2 * wastage_test["demand_avg_7"]
).astype(int)

print(classification_report(
    y_wastage_test,
    baseline_wastage_predictions
))

print("Confusion matrix:")
print(confusion_matrix(
    y_wastage_test,
    baseline_wastage_predictions
))

print("Precision:", precision_score(
    y_wastage_test,
    baseline_wastage_predictions
))

print("Recall:", recall_score(
    y_wastage_test,
    baseline_wastage_predictions
))

print("F1:", f1_score(
    y_wastage_test,
    baseline_wastage_predictions
))

In [ ]:
threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    predictions = (
        wastage_risk_scores >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(
            y_wastage_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_wastage_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_wastage_test,
            predictions,
            zero_division=0
        ),
        "alerts": predictions.sum()
    })

threshold_results = pd.DataFrame(threshold_results)

print(threshold_results)

In [ ]:
WASTAGE_THRESHOLD = 0.75

wastage_test["wastage_risk_score"] = wastage_risk_scores

wastage_test["wastage_alert"] = (
    wastage_test["wastage_risk_score"] >= WASTAGE_THRESHOLD
).astype(int)

print(
    wastage_test["wastage_alert"].value_counts()
)

In [ ]:
def get_wastage_level(score):
    if score >= 0.75:
        return "High"
    if score >= 0.50:
        return "Medium"
    return "Low"

wastage_test["wastage_risk_level"] = (
    wastage_test["wastage_risk_score"]
    .apply(get_wastage_level)
)

print(
    wastage_test["wastage_risk_level"].value_counts()
)

**Yes. Now we move from individual models to the actual decision engine.**

The architecture will be:

Demand Forecast
      ↓
Spike Risk ─────────┐
                    ↓
Inventory State → SHORTAGE / SURPLUS
                    ↓
Wastage Risk ───────┤
                    ↓
             Decision Engine
                    ↓
       Transfer / Replenishment
              Recommendation

**Saving Model 3**

In [ ]:
import joblib

joblib.dump(
    wastage_model,
    OUTPUT_DIR / "wastage_risk_model.pkl"
)

joblib.dump(
    wastage_preprocessor,
    OUTPUT_DIR / "wastage_risk_preprocessor.pkl"
)

print("Model 3 files saved:")
print("wastage_risk_model.pkl")
print("wastage_risk_preprocessor.pkl")

Creating the operational decision table$0

In [ ]:
final_df = decision_df.copy()

wastage_results = wastage_test[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ]
].copy()

wastage_results["wastage_risk"] = wastage_risk_scores

final_df = final_df.merge(
    wastage_results,
    on=[
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    how="left"
)

final_df["wastage_risk_level"] = final_df["wastage_risk"].apply(
    get_wastage_level
)

print(final_df.shape)
print(final_df.head())

In [ ]:
final_df["safety_stock"] = (
    final_df["predicted_demand"] * 2
)

final_df["surplus_units"] = (
    final_df["closing_inventory"] -
    final_df["safety_stock"]
).clip(lower=0)

final_df["deficit_units"] = (
    final_df["safety_stock"] -
    final_df["closing_inventory"]
).clip(lower=0)

print(
    final_df[
        [
            "blood_bank_id",
            "blood_group",
            "component_type",
            "closing_inventory",
            "predicted_demand",
            "safety_stock",
            "surplus_units",
            "deficit_units"
        ]
    ].head(10)
)

In [ ]:
final_df["potential_donor"] = (
    (final_df["surplus_units"] >= 5) &
    (final_df["wastage_risk"] >= 0.50)
).astype(int)

final_df["potential_receiver"] = (
    (final_df["deficit_units"] >= 1) |
    (final_df["risk_band"].isin(["High", "Very High"]))
).astype(int)

print("Potential donors:", final_df["potential_donor"].sum())
print("Potential receivers:", final_df["potential_receiver"].sum())

In [ ]:
final_df["potential_donor"] = (
    (final_df["surplus_units"] >= 5) &
    (final_df["wastage_risk"] >= 0.50)
).astype(int)

final_df["potential_receiver"] = (
    final_df["deficit_units"] >= 1
).astype(int)

print("Potential donors:", final_df["potential_donor"].sum())
print("Potential receivers:", final_df["potential_receiver"].sum())

In [ ]:
print("\nDonors by blood group:")
print(
    final_df[final_df["potential_donor"] == 1]
    ["blood_group"]
    .value_counts()
)

print("\nReceivers by blood group:")
print(
    final_df[final_df["potential_receiver"] == 1]
    ["blood_group"]
    .value_counts()
)

print("\nDonors by component:")
print(
    final_df[final_df["potential_donor"] == 1]
    ["component_type"]
    .value_counts()
)

print("\nReceivers by component:")
print(
    final_df[final_df["potential_receiver"] == 1]
    ["component_type"]
    .value_counts()
)

In [ ]:
donors = final_df[
    final_df["potential_donor"] == 1
].copy()

receivers = final_df[
    final_df["potential_receiver"] == 1
].copy()

print("Donor pool:", len(donors))
print("Receiver pool:", len(receivers))

In [ ]:
matches = []

for _, receiver in receivers.iterrows():

    candidates = donors[
        (donors["date"] == receiver["date"]) &
        (donors["blood_group"] == receiver["blood_group"]) &
        (donors["component_type"] == receiver["component_type"]) &
        (donors["blood_bank_id"] != receiver["blood_bank_id"])
    ].copy()

    if candidates.empty:
        continue

    donor = candidates.sort_values(
        "surplus_units",
        ascending=False
    ).iloc[0]

    transfer_quantity = min(
        receiver["deficit_units"],
        donor["surplus_units"]
    )

    if transfer_quantity <= 0:
        continue

    matches.append({
        "date": receiver["date"],
        "donor_bank": donor["blood_bank_id"],
        "receiver_bank": receiver["blood_bank_id"],
        "blood_group": receiver["blood_group"],
        "component_type": receiver["component_type"],
        "donor_surplus": donor["surplus_units"],
        "receiver_deficit": receiver["deficit_units"],
        "recommended_transfer": transfer_quantity,
        "donor_wastage_risk": donor["wastage_risk"],
        "receiver_risk_score": receiver["risk_score"]
    })

transfer_df = pd.DataFrame(matches)

print("Transfer recommendations:", len(transfer_df))
print(transfer_df.head(10))

In [ ]:
donor_balance = {}

for _, donor in donors.iterrows():
    key = (
        donor["date"],
        donor["blood_bank_id"],
        donor["blood_group"],
        donor["component_type"]
    )
    donor_balance[key] = donor["surplus_units"]

matches = []

receivers_sorted = receivers.sort_values(
    "deficit_units",
    ascending=False
)

for _, receiver in receivers_sorted.iterrows():

    candidates = donors[
        (donors["date"] == receiver["date"]) &
        (donors["blood_group"] == receiver["blood_group"]) &
        (donors["component_type"] == receiver["component_type"]) &
        (donors["blood_bank_id"] != receiver["blood_bank_id"])
    ].copy()

    candidates["available_surplus"] = candidates.apply(
        lambda row: donor_balance.get(
            (
                row["date"],
                row["blood_bank_id"],
                row["blood_group"],
                row["component_type"]
            ),
            0
        ),
        axis=1
    )

    candidates = candidates[
        candidates["available_surplus"] > 0
    ]

    if candidates.empty:
        continue

    donor = candidates.sort_values(
        ["wastage_risk", "available_surplus"],
        ascending=[False, False]
    ).iloc[0]

    donor_key = (
        donor["date"],
        donor["blood_bank_id"],
        donor["blood_group"],
        donor["component_type"]
    )

    transfer_quantity = min(
        receiver["deficit_units"],
        donor["available_surplus"]
    )

    if transfer_quantity <= 0:
        continue

    donor_balance[donor_key] -= transfer_quantity

    matches.append({
        "date": receiver["date"],
        "donor_bank": donor["blood_bank_id"],
        "receiver_bank": receiver["blood_bank_id"],
        "blood_group": receiver["blood_group"],
        "component_type": receiver["component_type"],
        "donor_surplus": donor["available_surplus"],
        "receiver_deficit": receiver["deficit_units"],
        "recommended_transfer": transfer_quantity,
        "donor_wastage_risk": donor["wastage_risk"],
        "receiver_risk_score": receiver["risk_score"]
    })

transfer_df = pd.DataFrame(matches)

print("Transfer recommendations:", len(transfer_df))
print(
    "Recommended units:",
    transfer_df["recommended_transfer"].sum()
)

In [ ]:
donor_usage = (
    transfer_df
    .groupby(
        [
            "date",
            "donor_bank",
            "blood_group",
            "component_type"
        ]
    )["recommended_transfer"]
    .sum()
    .reset_index()
)

donor_available = donors[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "surplus_units"
    ]
].rename(
    columns={
        "blood_bank_id": "donor_bank",
        "surplus_units": "available_surplus"
    }
)

donor_check = donor_usage.merge(
    donor_available,
    on=[
        "date",
        "donor_bank",
        "blood_group",
        "component_type"
    ],
    how="left"
)

donor_check["constraint_violation"] = (
    donor_check["recommended_transfer"] >
    donor_check["available_surplus"] + 1e-9
)

print("Constraint violations:")
print(donor_check["constraint_violation"].sum())

In [ ]:
transfer_df["recommended_transfer_units"] = (
    np.floor(transfer_df["recommended_transfer"])
).astype(int)

transfer_df = transfer_df[
    transfer_df["recommended_transfer_units"] >= 1
].copy()

print("Whole-unit transfer recommendations:", len(transfer_df))
print(
    "Total recommended units:",
    transfer_df["recommended_transfer_units"].sum()
)

In [ ]:
donor_usage = (
    transfer_df
    .groupby(
        [
            "date",
            "donor_bank",
            "blood_group",
            "component_type"
        ]
    )["recommended_transfer_units"]
    .sum()
    .reset_index()
)

donor_available = donors[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "surplus_units"
    ]
].rename(
    columns={
        "blood_bank_id": "donor_bank",
        "surplus_units": "available_surplus"
    }
)

donor_check = donor_usage.merge(
    donor_available,
    on=[
        "date",
        "donor_bank",
        "blood_group",
        "component_type"
    ],
    how="left"
)

donor_check["constraint_violation"] = (
    donor_check["recommended_transfer_units"] >
    donor_check["available_surplus"] + 1e-9
)

print(
    "Constraint violations after rounding:",
    donor_check["constraint_violation"].sum()
)

In [ ]:
receiver_usage = (
    transfer_df
    .groupby(
        [
            "date",
            "receiver_bank",
            "blood_group",
            "component_type"
        ]
    )["recommended_transfer_units"]
    .sum()
    .reset_index()
)

receiver_usage = receiver_usage.rename(
    columns={
        "recommended_transfer_units": "transfer_received"
    }
)

receiver_check = receivers[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "safety_stock",
        "deficit_units"
    ]
].rename(
    columns={
        "blood_bank_id": "receiver_bank"
    }
)

receiver_check = receiver_check.merge(
    receiver_usage,
    on=[
        "date",
        "receiver_bank",
        "blood_group",
        "component_type"
    ],
    how="left"
)

receiver_check["transfer_received"] = (
    receiver_check["transfer_received"]
    .fillna(0)
)

receiver_check["inventory_after_transfer"] = (
    receiver_check["closing_inventory"] +
    receiver_check["transfer_received"]
)

receiver_check["remaining_deficit"] = (
    receiver_check["safety_stock"] -
    receiver_check["inventory_after_transfer"]
).clip(lower=0)

receiver_check["resolved_by_transfer"] = (
    receiver_check["remaining_deficit"] == 0
)

print(
    "Receivers:",
    len(receiver_check)
)

print(
    "Receivers with a transfer:",
    (receiver_check["transfer_received"] > 0).sum()
)

print(
    "Deficits resolved:",
    receiver_check["resolved_by_transfer"].sum()
)

print(
    "Deficits remaining:",
    (~receiver_check["resolved_by_transfer"]).sum()
)

In [ ]:
receivers = final_df[
    final_df["potential_receiver"] == 1
].copy()

receivers["deficit_units"] = (
    np.ceil(receivers["deficit_units"])
).astype(int)

receivers = receivers[
    receivers["deficit_units"] >= 1
].copy()

print("Whole-unit receivers:", len(receivers))
print(
    "Total receiver deficit:",
    receivers["deficit_units"].sum()
)

In [ ]:
donor_balance = {}

for _, donor in donors.iterrows():
    key = (
        donor["date"],
        donor["blood_bank_id"],
        donor["blood_group"],
        donor["component_type"]
    )
    donor_balance[key] = int(np.floor(donor["surplus_units"]))

matches = []

receivers_sorted = receivers.sort_values(
    ["risk_score", "deficit_units"],
    ascending=[False, False]
)

for _, receiver in receivers_sorted.iterrows():

    candidates = donors[
        (donors["date"] == receiver["date"]) &
        (donors["blood_group"] == receiver["blood_group"]) &
        (donors["component_type"] == receiver["component_type"]) &
        (donors["blood_bank_id"] != receiver["blood_bank_id"])
    ].copy()

    candidates["available_surplus"] = candidates.apply(
        lambda row: donor_balance.get(
            (
                row["date"],
                row["blood_bank_id"],
                row["blood_group"],
                row["component_type"]
            ),
            0
        ),
        axis=1
    )

    candidates = candidates[
        candidates["available_surplus"] >= 1
    ]

    if candidates.empty:
        continue

    donor = candidates.sort_values(
        ["wastage_risk", "available_surplus"],
        ascending=[False, False]
    ).iloc[0]

    donor_key = (
        donor["date"],
        donor["blood_bank_id"],
        donor["blood_group"],
        donor["component_type"]
    )

    transfer_quantity = min(
        receiver["deficit_units"],
        donor["available_surplus"]
    )

    if transfer_quantity < 1:
        continue

    donor_balance[donor_key] -= transfer_quantity

    matches.append({
        "date": receiver["date"],
        "donor_bank": donor["blood_bank_id"],
        "receiver_bank": receiver["blood_bank_id"],
        "blood_group": receiver["blood_group"],
        "component_type": receiver["component_type"],
        "donor_surplus": donor["available_surplus"],
        "receiver_deficit": receiver["deficit_units"],
        "recommended_transfer_units": int(transfer_quantity),
        "donor_wastage_risk": donor["wastage_risk"],
        "receiver_risk_score": receiver["risk_score"]
    })

transfer_df = pd.DataFrame(matches)

print("Transfer recommendations:", len(transfer_df))
print(
    "Recommended units:",
    transfer_df["recommended_transfer_units"].sum()
)


In [ ]:
receiver_usage = (
    transfer_df
    .groupby(
        [
            "date",
            "receiver_bank",
            "blood_group",
            "component_type"
        ]
    )["recommended_transfer_units"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "recommended_transfer_units": "transfer_received"
        }
    )
)

receiver_check = receivers[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "safety_stock",
        "deficit_units"
    ]
].rename(
    columns={
        "blood_bank_id": "receiver_bank"
    }
)

receiver_check = receiver_check.merge(
    receiver_usage,
    on=[
        "date",
        "receiver_bank",
        "blood_group",
        "component_type"
    ],
    how="left"
)

receiver_check["transfer_received"] = (
    receiver_check["transfer_received"].fillna(0)
)

receiver_check["inventory_after_transfer"] = (
    receiver_check["closing_inventory"] +
    receiver_check["transfer_received"]
)

receiver_check["remaining_deficit"] = (
    receiver_check["safety_stock"] -
    receiver_check["inventory_after_transfer"]
).clip(lower=0)

receiver_check["resolved_by_transfer"] = (
    receiver_check["remaining_deficit"] < 1
)

print("Receivers:", len(receiver_check))
print(
    "Receivers with a transfer:",
    (receiver_check["transfer_received"] > 0).sum()
)
print(
    "Deficits resolved:",
    receiver_check["resolved_by_transfer"].sum()
)
print(
    "Deficits remaining:",
    (~receiver_check["resolved_by_transfer"]).sum()
)

In [ ]:
unresolved_receivers = receiver_check[
    ~receiver_check["resolved_by_transfer"]
].copy()

print("Unresolved receivers:", len(unresolved_receivers))

print(
    "Total remaining deficit:",
    unresolved_receivers["remaining_deficit"].sum()
)

print(
    "Average remaining deficit:",
    unresolved_receivers["remaining_deficit"].mean()
)

print("\nBy blood group:")
print(
    unresolved_receivers
    .groupby("blood_group")["remaining_deficit"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

print("\nBy component:")
print(
    unresolved_receivers
    .groupby("component_type")["remaining_deficit"]
    .agg(["count", "sum"])
    .sort_values("sum", ascending=False)
)

In [ ]:
receiver_check["transfer_status"] = np.select(
    [
        receiver_check["resolved_by_transfer"],
        receiver_check["transfer_received"] > 0
    ],
    [
        "Resolved",
        "Partially Resolved"
    ],
    default="Network Uncovered"
)

print(
    receiver_check["transfer_status"]
    .value_counts()
)

In [ ]:
status_summary = (
    receiver_check
    .groupby("transfer_status")
    .agg(
        receivers=("receiver_bank", "count"),
        transfer_units=("transfer_received", "sum"),
        remaining_deficit=("remaining_deficit", "sum")
    )
    .reset_index()
)

status_summary

**Build the final operational recommendation**
A single table that the frontend can directly consume.

In [ ]:
final_transfer_df = receiver_check[
    [
        "date",
        "receiver_bank",
        "blood_group",
        "component_type",
        "closing_inventory",
        "safety_stock",
        "deficit_units",
        "transfer_received",
        "inventory_after_transfer",
        "remaining_deficit",
        "transfer_status"
    ]
].copy()

transfer_lookup = transfer_df[
    [
        "date",
        "receiver_bank",
        "donor_bank",
        "blood_group",
        "component_type",
        "recommended_transfer_units",
        "donor_wastage_risk",
        "receiver_risk_score"
    ]
].copy()

final_transfer_df = final_transfer_df.merge(
    transfer_lookup,
    on=[
        "date",
        "receiver_bank",
        "blood_group",
        "component_type"
    ],
    how="left"
)

final_transfer_df["recommended_action"] = np.select(
    [
        final_transfer_df["transfer_status"] == "Resolved",
        final_transfer_df["transfer_status"] == "Partially Resolved",
        final_transfer_df["transfer_status"] == "Network Uncovered"
    ],
    [
        "Transfer recommended",
        "Transfer plus external replenishment required",
        "External replenishment required"
    ],
    default="Review"
)

print("Final recommendation rows:", len(final_transfer_df))

final_transfer_df.head(10)

In [ ]:
final_transfer_df["transfer_received"] = (
    final_transfer_df["transfer_received"]
    .round()
    .astype(int)
)

final_transfer_df["recommended_transfer_units"] = (
    final_transfer_df["recommended_transfer_units"]
    .fillna(0)
    .round()
    .astype(int)
)

final_transfer_df["donor_wastage_risk"] = (
    final_transfer_df["donor_wastage_risk"]
    .fillna(0)
)

final_transfer_df["receiver_risk_score"] = (
    final_transfer_df["receiver_risk_score"]
    .fillna(0)
)

print(final_transfer_df.dtypes)

In [ ]:
transfer_summary = {
    "receivers": len(receiver_check),
    "receivers_resolved": (
        receiver_check["transfer_status"] == "Resolved"
    ).sum(),
    "receivers_partial": (
        receiver_check["transfer_status"] == "Partially Resolved"
    ).sum(),
    "receivers_uncovered": (
        receiver_check["transfer_status"] == "Network Uncovered"
    ).sum(),
    "transfer_recommendations": len(transfer_df),
    "total_transfer_units": transfer_df[
        "recommended_transfer_units"
    ].sum(),
    "total_remaining_deficit": receiver_check[
        "remaining_deficit"
    ].sum()
}

for key, value in transfer_summary.items():
    print(f"{key}: {value}")

We have now built:

Model 1 — Demand

Predict next-day demand

→ regression
→ spike-risk classifier

Model 2 — Inventory

Identify shortage risk

→ ML risk score
→ rule-based inventory shortage logic

Model 3 — Wastage

Identify potential expiry/wastage

→ wastage-risk classifier

Decision Engine

Combine the outputs

→ risk score
→ safety stock
→ surplus/deficit
→ donor candidates
→ receiver candidates
→ constrained transfers
→ unresolved shortage detection

In [ ]:
network_decisions = final_df[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory",
        "predicted_demand",
        "predicted_coverage",
        "spike_risk",
        "inventory_risk",
        "wastage_risk",
        "risk_score",
        "risk_band"
    ]
].copy()

transfer_info = final_transfer_df[
    [
        "date",
        "receiver_bank",
        "blood_group",
        "component_type",
        "donor_bank",
        "recommended_transfer_units",
        "transfer_status",
        "inventory_after_transfer",
        "remaining_deficit",
        "recommended_action"
    ]
].copy()

transfer_info = transfer_info.rename(
    columns={
        "receiver_bank": "blood_bank_id"
    }
)

network_decisions = network_decisions.merge(
    transfer_info,
    on=[
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    how="left"
)

network_decisions["recommended_transfer_units"] = (
    network_decisions["recommended_transfer_units"]
    .fillna(0)
    .astype(int)
)

network_decisions["transfer_status"] = (
    network_decisions["transfer_status"]
    .fillna("No transfer required")
)

network_decisions["recommended_action"] = (
    network_decisions["recommended_action"]
    .fillna("No immediate action")
)

print("Network decision rows:", len(network_decisions))
print("Columns:", len(network_decisions.columns))

network_decisions.head(10)

In [ ]:
def get_operational_status(row):

    if row["predicted_demand"] > 0:
        if row["closing_inventory"] < row["predicted_demand"]:
            return "Critical Shortage"

        if row["predicted_coverage"] < 2:
            return "High Shortage Risk"

    if row["wastage_risk"] >= 0.75:
        return "High Wastage Risk"

    if row["spike_risk"] >= 0.30:
        return "Demand Spike Risk"

    if row["predicted_coverage"] < 5:
        return "Monitor Inventory"

    return "Stable"


network_decisions["operational_status"] = (
    network_decisions.apply(
        get_operational_status,
        axis=1
    )
)

print(
    network_decisions["operational_status"]
    .value_counts()
)

In [ ]:
operational_summary = (
    network_decisions
    .groupby("operational_status")
    .agg(
        records=("blood_bank_id", "count"),
        avg_predicted_demand=("predicted_demand", "mean"),
        avg_inventory=("closing_inventory", "mean"),
        avg_spike_risk=("spike_risk", "mean"),
        avg_wastage_risk=("wastage_risk", "mean")
    )
    .sort_values("records", ascending=False)
)

operational_summary

**
1. Model 1 — Demand Forecast
* predicts next-day demand
* regression + spike-risk classifier

2. Model 2 — Inventory Risk
* estimates shortage risk
* used as a supporting signal rather than the primary shortage decision

3. Model 3 — Wastage/Expiry Risk
* predicts next-day wastage/expiry risk

4. Decision Engine
* converts model outputs into operational states

5. Transfer Engine
*   identifies donor/receiver pairs under your explicit constraints
* whole-unit transfers
* no donor over-allocation
* same blood group
* same component
* same date
* different bank



6. Network-level outcomes
* Resolved
* Partially Resolved
* Network Uncovered
* External replenishment requirement**



Saving Complete Outputs

In [ ]:
network_decisions.to_csv(
    OUTPUT_DIR / "network_decisions.csv",
    index=False
)

final_transfer_df.to_csv(
    OUTPUT_DIR / "transfer_recommendations.csv",
    index=False
)

operational_summary.to_csv(
    OUTPUT_DIR / "operational_summary.csv"
)

print("Saved:")
print(OUTPUT_DIR / "network_decisions.csv")
print(OUTPUT_DIR / "transfer_recommendations.csv")
print(OUTPUT_DIR / "operational_summary.csv")

In [ ]:
model_metrics = pd.DataFrame([
    {
        "model": "Demand Forecast",
        "task": "Regression",
        "MAE": 2.0235,
        "RMSE": 4.1174
    },
    {
        "model": "Spike Risk",
        "task": "Classification",
        "ROC_AUC": 0.9384,
        "PR_AUC": np.nan
    },
    {
        "model": "Inventory Risk",
        "task": "Classification",
        "ROC_AUC": 0.9606,
        "PR_AUC": 0.4668
    },
    {
        "model": "Wastage Risk",
        "task": "Classification",
        "ROC_AUC": 0.9186,
        "PR_AUC": 0.4746
    }
])

model_metrics.to_csv(
    OUTPUT_DIR / "model_metrics.csv",
    index=False
)

model_metrics

In [ ]:
print("\nOutput files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("smartblood_outputs", "zip", "outputs")
files.download("smartblood_outputs.zip")

# **=============================================**
# **Synthetic Live-Data Simulation**
Further testing.

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import joblib
from pathlib import Path

OUTPUT_DIR = Path("/")
OUTPUT_DIR.mkdir(exist_ok=True)

RNG = np.random.default_rng(42)

print("Environment ready")

In [ ]:
import joblib
from pathlib import Path

OUTPUT_DIR = Path("")

demand_model = joblib.load(
    OUTPUT_DIR / "demand_forecast_model.pkl"
)

demand_preprocessor = joblib.load(
    OUTPUT_DIR / "demand_forecast_preprocessor.pkl"
)

spike_model = joblib.load(
    OUTPUT_DIR / "spike_risk_model.pkl"
)

spike_preprocessor = joblib.load(
    OUTPUT_DIR / "spike_risk_preprocessor.pkl"
)

inventory_model = joblib.load(
    OUTPUT_DIR / "inventory_risk_model.pkl"
)

inventory_preprocessor = joblib.load(
    OUTPUT_DIR / "inventory_risk_preprocessor.pkl"
)

wastage_model = joblib.load(
    OUTPUT_DIR / "wastage_risk_model.pkl"
)

wastage_preprocessor = joblib.load(
    OUTPUT_DIR / "wastage_risk_preprocessor.pkl"
)

print("All models loaded successfully.")

In [ ]:
simulation_start = pd.Timestamp("2024-09-30")

simulation_state = smart[
    smart["date"] == simulation_start
].copy()

simulation_state = simulation_state[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "closing_inventory"
    ]
].copy()

print(simulation_state.shape)
simulation_state.head()

In [ ]:
historical_demand = smart[
    smart["date"] <= simulation_start
].copy()

historical_demand["day_of_week"] = (
    historical_demand["date"].dt.dayofweek
)

demand_distribution = (
    historical_demand
    .groupby(
        [
            "blood_bank_id",
            "blood_group",
            "component_type",
            "day_of_week"
        ]
    )["units_requested"]
    .apply(list)
    .to_dict()
)

print("Demand distributions:", len(demand_distribution))

In [ ]:
RNG = np.random.default_rng(42)

def generate_daily_demand(date, state):

    rows = []

    day_of_week = date.dayofweek

    for _, row in state.iterrows():

        key = (
            row["blood_bank_id"],
            row["blood_group"],
            row["component_type"],
            day_of_week
        )

        values = demand_distribution.get(key)

        if values is None or len(values) == 0:
            demand = 0
        else:
            demand = RNG.choice(values)

        rows.append({
            "date": date,
            "blood_bank_id": row["blood_bank_id"],
            "blood_group": row["blood_group"],
            "component_type": row["component_type"],
            "units_requested": int(demand)
        })

    return pd.DataFrame(rows)

In [ ]:
synthetic_day = generate_daily_demand(
    pd.Timestamp("2024-10-01"),
    simulation_state
)

synthetic_day.head()
synthetic_day.shape

In [ ]:
def apply_simulation_event(df, date):

    df = df.copy()

    if date == pd.Timestamp("2024-10-10"):
        mask = (
            (df["component_type"] == "PRBC") &
            (df["blood_group"].isin(["O+", "A+"]))
        )
        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 3
        ).astype(int)

    if date == pd.Timestamp("2024-10-20"):
        mask = df["component_type"] == "Platelets"
        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 2
        ).astype(int)

    if date == pd.Timestamp("2024-11-01"):
        mask = df["blood_bank_id"] == "BB005"
        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 2
        ).astype(int)

    return df

In [ ]:
simulation_days = []

state = simulation_state.copy()

dates = pd.date_range(
    start="2024-10-01",
    end="2024-10-30",
    freq="D"
)

for date in dates:

    day = generate_daily_demand(
        date,
        state
    )

    day = apply_simulation_event(
        day,
        date
    )

    simulation_days.append(day)

synthetic_stream = pd.concat(
    simulation_days,
    ignore_index=True
)

print(synthetic_stream.shape)
synthetic_stream.head()

In [ ]:
simulation_history = smart[
    smart["date"] <= simulation_start
].copy()

simulation_history["day_of_week"] = (
    simulation_history["date"].dt.dayofweek
)

flow_cols = [
    "donations_collected",
    "transfers_in",
    "transfers_out",
    "units_wasted",
    "units_expired"
]

flow_distributions = {}

for col in flow_cols:
    flow_distributions[col] = (
        simulation_history
        .groupby(
            [
                "blood_bank_id",
                "blood_group",
                "component_type",
                "day_of_week"
            ]
        )[col]
        .apply(list)
        .to_dict()
    )

print("Flow distributions created.")

In [ ]:
def sample_flow(distribution, key):

    values = distribution.get(key)

    if values is None or len(values) == 0:
        return 0

    return int(RNG.choice(values))

In [ ]:
def adjust_donations(donations, inventory, demand):

    if inventory > demand * 10:
        donations = int(donations * 0.25)

    elif inventory > demand * 5:
        donations = int(donations * 0.50)

    return max(donations, 0)

def generate_complete_day(date, state):

    rows = []

    day_of_week = date.dayofweek

    for _, row in state.iterrows():

        key = (
            row["blood_bank_id"],
            row["blood_group"],
            row["component_type"],
            day_of_week
        )

        demand = sample_flow(
            demand_distribution,
            key
        )

        donations = sample_flow(
          flow_distributions["donations_collected"],
          key
        )

        donations = adjust_donations(
            donations,
            row["closing_inventory"],
            demand
        )

        transfers_in = sample_flow(
            flow_distributions["transfers_in"],
            key
        )

        transfers_out = sample_flow(
            flow_distributions["transfers_out"],
            key
        )

        wasted = sample_flow(
            flow_distributions["units_wasted"],
            key
        )

        expired = sample_flow(
            flow_distributions["units_expired"],
            key
        )

        opening_inventory = int(
            row["closing_inventory"]
        )

        available_inventory = max(
            0,
            opening_inventory
            + donations
            + transfers_in
            - transfers_out
        )

        fulfilled = min(
            demand,
            available_inventory
        )

        closing_inventory = max(
            0,
            available_inventory
            - fulfilled
            - wasted
            - expired
        )

        rows.append({
            "date": date,
            "blood_bank_id": row["blood_bank_id"],
            "blood_group": row["blood_group"],
            "component_type": row["component_type"],
            "opening_inventory": opening_inventory,
            "donations_collected": donations,
            "transfers_in": transfers_in,
            "transfers_out": transfers_out,
            "units_requested": demand,
            "units_fulfilled": fulfilled,
            "units_expired": expired,
            "units_wasted": wasted,
            "closing_inventory": closing_inventory
        })

    return pd.DataFrame(rows)

In [ ]:
def apply_simulation_event(df, date):

    df = df.copy()

    if date == pd.Timestamp("2024-10-10"):
        mask = (
            (df["component_type"] == "PRBC") &
            (df["blood_group"].isin(["O+", "A+"]))
        )

        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 3
        ).astype(int)

    if date == pd.Timestamp("2024-10-20"):
        mask = (
            df["component_type"] == "Platelets"
        )

        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 2
        ).astype(int)

    if date == pd.Timestamp("2024-11-01"):
        mask = (
            df["blood_bank_id"] == "BB005"
        )

        df.loc[mask, "units_requested"] = (
            df.loc[mask, "units_requested"] * 2
        ).astype(int)

    if date == pd.Timestamp("2024-11-10"):
        mask = (
            df["blood_bank_id"] == "BB003"
        )

        df.loc[mask, "donations_collected"] = (
            df.loc[mask, "donations_collected"] * 0.2
        ).astype(int)

    return df

In [ ]:
def generate_simulated_day(date, state):

    day = generate_complete_day(
        date,
        state
    )

    day = apply_simulation_event(
        day,
        date
    )

    day["available_inventory"] = (
        day["opening_inventory"]
        + day["donations_collected"]
        + day["transfers_in"]
        - day["transfers_out"]
    ).clip(lower=0)

    day["units_fulfilled"] = np.minimum(
        day["units_requested"],
        day["available_inventory"]
    )

    day["closing_inventory"] = (
        day["available_inventory"]
        - day["units_fulfilled"]
        - day["units_expired"]
        - day["units_wasted"]
    ).clip(lower=0)

    return day

In [ ]:
simulation_dates = pd.date_range(
    start="2024-10-01",
    periods=60,
    freq="D"
)

state = simulation_state.copy()

simulation_results = []

for date in simulation_dates:

    day = generate_simulated_day(
        date,
        state
    )

    simulation_results.append(day)

    state = day[
        [
            "date",
            "blood_bank_id",
            "blood_group",
            "component_type",
            "closing_inventory"
        ]
    ].copy()

synthetic_live_data = pd.concat(
    simulation_results,
    ignore_index=True
)

print(
    "Simulation shape:",
    synthetic_live_data.shape
)

print(
    "Date range:",
    synthetic_live_data["date"].min(),
    "to",
    synthetic_live_data["date"].max()
)

In [ ]:
synthetic_live_data[
    [
        "units_requested",
        "units_fulfilled",
        "donations_collected",
        "transfers_in",
        "transfers_out",
        "units_wasted",
        "units_expired",
        "closing_inventory"
    ]
].describe()

In [ ]:
print(
    "Total requested:",
    synthetic_live_data["units_requested"].sum()
)

print(
    "Total fulfilled:",
    synthetic_live_data["units_fulfilled"].sum()
)

print(
    "Fulfillment rate:",
    synthetic_live_data["units_fulfilled"].sum()
    / synthetic_live_data["units_requested"].sum()
)

In [ ]:
print(
    "Rows with shortages:",
    (
        synthetic_live_data["units_requested"]
        >
        synthetic_live_data["units_fulfilled"]
    ).sum()
)

In [ ]:
sim = synthetic_live_data.copy()

sim["day_of_week"] = sim["date"].dt.dayofweek
sim["month"] = sim["date"].dt.month
sim["day_of_year"] = sim["date"].dt.dayofyear
sim["week_of_year"] = (
    sim["date"].dt.isocalendar().week.astype(int)
)
sim["is_weekend"] = (
    sim["day_of_week"] >= 5
).astype(int)

sim["target_date"] = (
    sim["date"] + pd.Timedelta(days=1)
)

sim["target_day_of_week"] = (
    sim["target_date"].dt.dayofweek
)

sim["target_month"] = (
    sim["target_date"].dt.month
)

sim["target_day_of_year"] = (
    sim["target_date"].dt.dayofyear
)

sim["target_week_of_year"] = (
    sim["target_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

sim["target_is_weekend"] = (
    sim["target_day_of_week"] >= 5
).astype(int)

In [ ]:
sim_group = sim.groupby(
    [
        "blood_bank_id",
        "blood_group",
        "component_type"
    ],
    group_keys=False
)

sim["demand_lag_1"] = (
    sim_group["units_requested"].shift(1)
)

sim["demand_lag_7"] = (
    sim_group["units_requested"].shift(7)
)

sim["demand_lag_14"] = (
    sim_group["units_requested"].shift(14)
)

sim["demand_lag_28"] = (
    sim_group["units_requested"].shift(28)
)

sim["demand_avg_3"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(3).mean()
    )
)

sim["demand_avg_7"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["demand_avg_14"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(14).mean()
    )
)

sim["demand_avg_28"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(28).mean()
    )
)

sim["demand_max_7"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).max()
    )
)

sim["demand_max_14"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(14).max()
    )
)

sim["demand_max_28"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(28).max()
    )
)

sim["demand_std_7"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).std()
    )
)

sim["spike_count_7"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(7)
        .apply(
            lambda y: (y >= 10).sum()
        )
    )
)

sim["spike_count_14"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(14)
        .apply(
            lambda y: (y >= 10).sum()
        )
    )
)

sim["spike_count_28"] = (
    sim_group["units_requested"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(28)
        .apply(
            lambda y: (y >= 10).sum()
        )
    )
)

sim["demand_trend"] = (
    sim["demand_avg_3"]
    - sim["demand_avg_14"]
)

In [ ]:
sim["fulfillment_rate"] = np.where(
    sim["units_requested"] > 0,
    sim["units_fulfilled"]
    / sim["units_requested"],
    1.0
)

sim["inventory_lag_1"] = (
    sim_group["closing_inventory"].shift(1)
)

sim["inventory_lag_7"] = (
    sim_group["closing_inventory"].shift(7)
)

sim["inventory_avg_7"] = (
    sim_group["closing_inventory"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["fulfillment_avg_7"] = (
    sim_group["fulfillment_rate"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["donations_avg_7"] = (
    sim_group["donations_collected"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["transfers_in_avg_7"] = (
    sim_group["transfers_in"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["transfers_out_avg_7"] = (
    sim_group["transfers_out"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["wastage_avg_7"] = (
    sim_group["units_wasted"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

sim["expiry_avg_7"] = (
    sim_group["units_expired"]
    .transform(
        lambda x:
        x.shift(1).rolling(7).mean()
    )
)

In [ ]:
sim["actual_next_day_demand"] = (
    sim_group["units_requested"].shift(-1)
)

In [ ]:
simulation_model_df = sim[
    feature_cols
    + [
        "date",
        "actual_next_day_demand"
    ]
].dropna().copy()

print(
    simulation_model_df.shape
)

In [ ]:
X_sim_demand = simulation_model_df[
    feature_cols
]

y_sim_demand = simulation_model_df[
    "actual_next_day_demand"
]

X_sim_demand_encoded = (
    demand_preprocessor.transform(
        X_sim_demand
    )
)

sim_demand_predictions = (
    demand_model.predict(
        X_sim_demand_encoded
    )
)

sim_demand_predictions = np.maximum(
    sim_demand_predictions,
    0
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

sim_mae = mean_absolute_error(
    y_sim_demand,
    sim_demand_predictions
)

sim_rmse = np.sqrt(
    mean_squared_error(
        y_sim_demand,
        sim_demand_predictions
    )
)

print("Synthetic Live MAE:", round(sim_mae, 4))
print("Synthetic Live RMSE:", round(sim_rmse, 4))

In [ ]:
simulation_spike_df = sim[
    spike_feature_cols
    + [
        "date",
        "actual_next_day_demand"
    ]
].dropna().copy()

simulation_spike_df["actual_spike"] = (
    simulation_spike_df[
        "actual_next_day_demand"
    ] >= 10
).astype(int)

In [ ]:
X_sim_spike = simulation_spike_df[
    spike_feature_cols
]

X_sim_spike_encoded = (
    spike_preprocessor.transform(
        X_sim_spike
    )
)

sim_spike_scores = (
    spike_model.predict_proba(
        X_sim_spike_encoded
    )[:, 1]
)

sim_spike_predictions = (
    sim_spike_scores >= 0.30
).astype(int)

In [ ]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)

print(
    classification_report(
        simulation_spike_df["actual_spike"],
        sim_spike_predictions
    )
)

print(
    "ROC-AUC:",
    round(
        roc_auc_score(
            simulation_spike_df["actual_spike"],
            sim_spike_scores
        ),
        4
    )
)

In [ ]:
event_dates = [
    pd.Timestamp("2024-10-10"),
    pd.Timestamp("2024-10-20"),
    pd.Timestamp("2024-11-01"),
    pd.Timestamp("2024-11-10")
]

event_results = simulation_spike_df[
    simulation_spike_df["date"].isin(event_dates)
].copy()

event_results[
    [
        "date",
        "blood_bank_id",
        "blood_group",
        "component_type",
        "actual_next_day_demand",
        "actual_spike"
    ]
].head(30)

In [ ]:
event_results["spike_risk_score"] = sim_spike_scores[
    simulation_spike_df["date"].isin(event_dates)
]

event_results.sort_values(
    "spike_risk_score",
    ascending=False
).head(30)

In [ ]:
def adjust_donations(
    donations,
    inventory,
    demand
):

    if inventory > demand * 10:
        donations = int(
            donations * 0.25
        )

    elif inventory > demand * 5:
        donations = int(
            donations * 0.50
        )

    return max(
        donations,
        0
    )